In [ ]:
import os
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

sys.path.append("../")
from detect_align import align_face
from src.utils import draw_bboxes_and_keypoints
from src.yolov5 import adjust_boxes_and_kpts, non_max_suppression_onnx, rescale_coordinates

%load_ext autoreload
%autoreload 2

In [ ]:
# Read image
img_p = "../data/img1.jpg"
img = cv2.imread(img_p)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.show()

In [ ]:
# Detect faces
model_fp = "../checkpoints/yolo/yolov5n-face.onnx"
model = cv2.dnn.readNetFromONNX(model_fp)

# Preprocess image for ONNX model
blob = cv2.dnn.blobFromImage(
    img,
    1 / 255.0,
    (640, 640),
    swapRB=True,
    crop=False,
)
print("Blob shape:", blob.shape)

# Set input and run inference
model.setInput(blob)
outputs = model.forward()
print("ONNX output shape:", outputs.shape)
preds_raw = outputs[0]

# Apply NMS
preds = non_max_suppression_onnx(preds_raw, conf_thres=0.25, iou_thresh=0.45)
print("Predictions after NMS:", preds.shape)

# Rescale coordinates
preds = rescale_coordinates(preds, img)
boxes = np.array(preds[:, :4], dtype=int)
keypoints = np.array(preds[:, 5:15], dtype=int)
bboxes, keypoints_all = adjust_boxes_and_kpts(boxes, keypoints)


img_out = draw_bboxes_and_keypoints(img, bboxes, keypoints_all)
plt.imshow(cv2.cvtColor(img_out, cv2.COLOR_BGR2RGB))
plt.show()

In [ ]:
# Align detected faces
aligned_faces = []
for i in range(len(keypoints_all)):
    keypoints = keypoints_all[i]
    landmarks = np.array(list(keypoints.values()), dtype=np.float32)

    img_aligned = align_face(img, landmarks)
    aligned_faces.append(img_aligned)
    # plt.show()

# Show aligned faces
fig, axs = plt.subplots(1, len(aligned_faces))
axs = np.ravel(axs)
axs: list[plt.Axes]
for i in range(len(aligned_faces)):
    axs[i].imshow(cv2.cvtColor(aligned_faces[i], cv2.COLOR_BGR2RGB))
    axs[i].axis("off")
fig.tight_layout()
fig.suptitle(
    "Detected and aligned faces:",
    y=0.7,  # Position title closer to plots
)
plt.show()